# Отдельный мост выполнения Python для Notebook UI

Этот блокнот поднимает временный защищённый endpoint в текущей сессии Google Colab. Он нужен только для выполнения ячеек; ИИ-учитель и его ключи остаются в Notebook UI.

Перед запуском изучите загруженный `.ipynb`: его код будет выполняться в вашей сессии Colab. Не закрывайте эту вкладку, пока используете подключение. Адрес туннеля и токен подключения действуют только пока работает сессия.

In [ ]:
!pip -q install flask

In [ ]:
import os, re, secrets, subprocess, threading, time, urllib.request

cloudflared_path = "/content/cloudflared"
if not os.path.exists(cloudflared_path):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared_path,
    )
    os.chmod(cloudflared_path, 0o755)
print("Туннельная утилита готова")

In [ ]:
from flask import Flask, jsonify, request
from IPython import get_ipython
from IPython.utils.capture import capture_output

bridge_token = secrets.token_urlsafe(32)
execution_lock = threading.Lock()
bridge_app = Flask("notebook_ui_colab_execution_bridge")

@bridge_app.get("/health")
def bridge_health():
    if not secrets.compare_digest(request.headers.get("X-Colab-Bridge-Token", ""), bridge_token):
        return jsonify({"error": "Unauthorized"}), 401
    return jsonify({"status": "ok"})

@bridge_app.post("/execute")
def bridge_execute():
    if not secrets.compare_digest(request.headers.get("X-Colab-Bridge-Token", ""), bridge_token):
        return jsonify({"error": "Unauthorized"}), 401
    body = request.get_json(silent=True) or {}
    code = body.get("code")
    if not isinstance(code, str) or len(code) > 1_000_000:
        return jsonify({"status": "error", "stdout": "", "stderr": "Некорректный код ячейки"}), 400
    try:
        with execution_lock, capture_output() as captured:
            result = get_ipython().run_cell(code, store_history=False)
        stderr = captured.stderr or ""
        if result.error_before_exec or result.error_in_exec:
            error = result.error_before_exec or result.error_in_exec
            stderr = (stderr + "\n" + str(error)).strip()
            return jsonify({"status": "error", "stdout": captured.stdout or "", "stderr": stderr})
        stdout = captured.stdout or ""
        if result.result is not None:
            stdout = (stdout + "\n" + repr(result.result)).strip()
        return jsonify({"status": "ok", "stdout": stdout, "stderr": stderr})
    except Exception as error:
        return jsonify({"status": "error", "stdout": "", "stderr": str(error)})

def start_bridge_server():
    bridge_app.run(host="0.0.0.0", port=5000, threaded=True, use_reloader=False)

threading.Thread(target=start_bridge_server, daemon=True).start()
time.sleep(2)
print("Мост выполнения запущен")

In [ ]:
tunnel = subprocess.Popen(
    [cloudflared_path, "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
tunnel_url = None
deadline = time.time() + 40
for line in iter(tunnel.stdout.readline, ""):
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    if time.time() >= deadline:
        break
if not tunnel_url:
    raise RuntimeError("Cloudflare Tunnel не запустился. Перезапустите эту ячейку.")
print("Адрес Colab: " + tunnel_url)
print("Токен подключения: " + bridge_token)
print("Вставьте оба значения в Notebook UI. Не публикуйте их.")